## limpeza e padronização do IDHM 

Importação de bibliotecas

In [3]:
import pandas as pd
import numpy as np
import duckdb
import unicodedata
from pathlib import Path
import warnings

# Suprimir warnings
warnings.filterwarnings('ignore')

# Configurações Visuais
pd.set_option('display.max_columns', None)

# Caminhos (Atualizado para IDHM.xlsx)
RAW_FILE = Path('../data/raw/atlas/IDHM.xlsx') 
PROCESSED_PATH = Path('../data/processed')
PROCESSED_PATH.mkdir(exist_ok=True, parents=True)

print("Ambiente configurado.")

Ambiente configurado.


Leitura e seleção de colunas 

In [4]:
def carregar_idhm(arquivo):
    print(f'Carregando arquivo: {arquivo}')
    
    try:
        # Tenta ler como Excel (padrao) ou CSV se falhar
        try:
            df = pd.read_excel(arquivo)
        except:
            df = pd.read_csv(arquivo, sep=',', encoding='utf-8')
    except Exception as e:
        print(f'[ERRO] Falha ao ler arquivo: {e}')
        return pd.DataFrame()
        
    # Primeira coluna e sempre o local
    col_local = df.columns[0] 
    print(f"Coluna de local identificada: {col_local}")
    
    dfs_anos = []
    
    # Mapeamento Explicito das colunas que voce confirmou
    mapa_anos = {
        2000: {
            'IDHM 2000': 'idhm',
            'IDHM Renda 2000': 'idhm_renda',
            'IDHM Longevidade 2000': 'idhm_longevidade',
            'IDHM Educação 2000': 'idhm_educ',
            'Renda per capita 2000': 'renda_pc',
            'Taxa de analfabetismo - 18 anos ou mais de idade 2000': 'tx_analfabetismo'
        },
        2010: {
            'IDHM 2010': 'idhm',
            'IDHM Renda 2010': 'idhm_renda',
            'IDHM Longevidade 2010': 'idhm_longevidade',
            'IDHM Educação 2010': 'idhm_educ',
            'Renda per capita 2010': 'renda_pc',
            'Taxa de analfabetismo - 18 anos ou mais de idade 2010': 'tx_analfabetismo'
        }
    }
    
    for ano, cols_alvo in mapa_anos.items():
        # Verifica quais colunas existem no arquivo
        cols_existentes = [c for c in cols_alvo.keys() if c in df.columns]
        
        if cols_existentes:
            # Seleciona Local + Colunas do Ano
            subset = df[[col_local] + cols_existentes].copy()
            
            # Renomeia para padrao limpo
            subset = subset.rename(columns=cols_alvo)
            subset = subset.rename(columns={col_local: 'nome_origem'})
            
            # Adiciona coluna de ano
            subset['ano'] = ano
            
            dfs_anos.append(subset)
            print(f"   Ano {ano} processado com {len(cols_existentes)} indicadores.")
            
    if dfs_anos:
        return pd.concat(dfs_anos, ignore_index=True)
    
    print("[ERRO] Nenhuma coluna esperada encontrada.")
    return pd.DataFrame()

df_idhm_raw = carregar_idhm(RAW_FILE)
print(f'Shape inicial: {df_idhm_raw.shape}')
df_idhm_raw.head()

Carregando arquivo: ../data/raw/atlas/IDHM.xlsx
Coluna de local identificada: Territorialidades
   Ano 2000 processado com 6 indicadores.
   Ano 2010 processado com 6 indicadores.
Shape inicial: (11138, 8)


,nome_origem,idhm,idhm_renda,idhm_longevidade,idhm_educ,renda_pc,tx_analfabetismo,ano
0,Brasil,0.612,0.692,0.727,0.456,592.46,14.50,2000
1,Abadia de Goiás (GO),0.569,0.623,0.765,0.386,385.66,13.08,2000
2,Abadia dos Dourados (MG),0.575,0.616,0.799,0.387,370.42,14.39,2000
3,Abadiânia (GO),0.503,0.598,0.730,0.292,330.07,18.94,2000
4,Abaeté (MG),0.587,0.664,0.792,0.385,498.82,13.28,2000


Tratamento de Código (Com Normalização de Acentos)

In [8]:
# Funcao para remover acentos (Normalizacao)
def normalizar(texto):
    if pd.isna(texto): return ""
    # Normaliza unicode (tira acento)
    nfkd = unicodedata.normalize('NFD', str(texto))
    # Tira caracteres de combinação, joga pra maiusculo e tira espacos extras
    return "".join([c for c in nfkd if not unicodedata.combining(c)]).upper().strip()

print("Cruzando nomes com codigos IBGE...")

# 1. Carregar Dicionario Confiavel (Do arquivo de populacao)
arquivo_ref = PROCESSED_PATH / 'populacao_completa.parquet'

if not arquivo_ref.exists():
    print("[ERRO] Arquivo populacao_completa.parquet nao encontrado.")
else:
    df_ref = pd.read_parquet(arquivo_ref)
    
    # Identificar coluna de nome (pode ser nome_origem ou nome_municipio)
    col_nome_ref = 'nome_origem' if 'nome_origem' in df_ref.columns else 'nome_municipio'
    print(f"Usando coluna de referencia: {col_nome_ref}")
    
    # Criar Dicionario usando o NOME COMPLETO (Com UF)
    # Isso garante que "Bom Jesus (RS)" nao misture com "Bom Jesus (RN)"
    df_ref_unique = df_ref[['codmun', col_nome_ref]].drop_duplicates()
    df_ref_unique['chave'] = df_ref_unique[col_nome_ref].apply(normalizar)
    
    dic_codigos = dict(zip(df_ref_unique['chave'], df_ref_unique['codmun']))
    
    # 2. Aplicar no IDHM
    # AQUI MUDOU: Nao fazemos mais o .split(' ('). Mantemos o nome inteiro.
    df_idhm_raw['chave'] = df_idhm_raw['nome_origem'].apply(normalizar)
    
    # Mapear codigo
    df_idhm_raw['codmun'] = df_idhm_raw['chave'].map(dic_codigos)
    
    # Verificar falhas
    nulos = df_idhm_raw['codmun'].isnull().sum()
    print(f"Cidades sem codigo encontrado: {nulos}")
    
    if nulos > 0:
        print("Exemplos sem codigo:", df_idhm_raw[df_idhm_raw['codmun'].isnull()]['nome_origem'].unique()[:5])
    
    # Limpar quem nao tem codigo
    df_idhm_clean = df_idhm_raw.dropna(subset=['codmun']).copy()
    print(f"Linhas restantes para processar: {len(df_idhm_clean)}")

Cruzando nomes com codigos IBGE...
Usando coluna de referencia: nome_origem
Cidades sem codigo encontrado: 20
Exemplos sem codigo: ['Brasil' 'Belém do São Francisco (PE)' 'Brasópolis (MG)' 'Embu (SP)'
 'Iguaraci (PE)']
Linhas restantes para processar: 11118


Interpolação e Projeção (2007 e 2015)

In [9]:
ANOS_ALVO = [2007, 2010, 2015]

dfs_finais = []

# Colunas numericas para interpolar
cols_num = ['idhm', 'idhm_renda', 'idhm_longevidade', 'idhm_educ', 'renda_pc', 'tx_analfabetismo']
cols_num = [c for c in cols_num if c in df_idhm_clean.columns]

print("Interpolando dados faltantes...")

for cod, dados in df_idhm_clean.groupby('codmun'):
    # Criar linha do tempo
    timeline = pd.DataFrame({'ano': ANOS_ALVO})
    merged = pd.merge(timeline, dados, on='ano', how='outer').sort_values('ano')
    
    # Preencher fixos
    merged['codmun'] = cod
    
    # Matematica:
    # 1. Interpola 2007 (Linear entre 2000 e 2010)
    merged[cols_num] = merged[cols_num].interpolate(method='linear')
    
    # 2. Projeta 2015 (Repete 2010 - Forward Fill)
    merged[cols_num] = merged[cols_num].ffill().bfill()
    
    # Filtra apenas anos de interesse
    final = merged[merged['ano'].isin(ANOS_ALVO)].copy()
    dfs_finais.append(final)

# Consolidar
df_idhm_final = pd.concat(dfs_finais, ignore_index=True)
print(f"IDHM Finalizado. Anos presentes: {df_idhm_final['ano'].unique()}")
print(f"Shape final: {df_idhm_final.shape}")

Interpolando dados faltantes...
IDHM Finalizado. Anos presentes: [2007 2010 2015]
Shape final: (16113, 11)


Salvar e validar 

In [10]:
outfile = PROCESSED_PATH / 'idhm_final.parquet'
df_idhm_final.to_parquet(outfile, index=False)
print(f'[SUCESSO] Arquivo salvo em: {outfile}')

# Validacao SQL
print('\n--- Validacao DuckDB ---')
con = duckdb.connect()
query = f"""
SELECT 
    ano, 
    COUNT(*) as qtd_cidades,
    AVG(idhm)::DECIMAL(10,3) as media_idhm,
    MIN(idhm)::DECIMAL(10,3) as min_idhm,
    MAX(idhm)::DECIMAL(10,3) as max_idhm
FROM '{outfile}'
GROUP BY ano
ORDER BY ano
"""
print(con.execute(query).df())

[SUCESSO] Arquivo salvo em: ../data/processed/idhm_final.parquet

--- Validacao DuckDB ---
    ano  qtd_cidades  media_idhm  min_idhm  max_idhm
0  2007         5277       0.525     0.208     0.820
1  2010         5559       0.659     0.418     0.862
2  2015         5277       0.660     0.418     0.862
